<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">4. Working with Complex Data</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 4.1 Lecture - Working with Complex Data
### Duration - 18 Minutes

> *Structured data beyond simple types*

This lecture covers Spark's complex data types - arrays, structs, and maps - including converting JSON strings to structured types, the functions for manipulating arrays, aggregating values into collections, and the best practices that keep complex data operations performant.

## Learning Objectives

**By the end of this lecture, you will be able to:**

- Describe Apache Spark's support for arrays, maps, and structs, and when these data types commonly appear in real-world scenarios
- Use functions for creating collections (`collect_list()`, `collect_set()`) in aggregation pipelines
- Manipulate and transform array-type columns using functions like `explode()`, `array_contains()`, and `element_at()`
- Work effectively with nested struct types, including accessing fields and flattening complex structures

## A. Complex Data Types in Spark

Complex data types are common in modern data pipelines, especially when working with JSON data, nested analytics results, or feature engineering:

| Type | PySpark Type | Description |
|------|--------------|-------------|
| __Array__ | `pyspark.sql.types.ArrayType` | Ordered collection of elements (e.g. `[1, 2, 3]`) |
| __Struct__ | `pyspark.sql.types.StructType` | Nested structure with predefined named fields |
| __Map__ | `pyspark.sql.types.MapType` | Key-value pairs (keys are not predefined) |

## B. JSON Strings vs Structs

<p style="font-size: 1em; line-height: 1.6; color: #333">Semi-structured data often arrives as JSON strings. Convert it early:</p>

- JSON strings require parsing overhead and waste memory
- Structs give type safety and better performance
- Convert early using `from_json()` with a schema

Given this example input data:

```
+---+--------------------------------------------+
|id |json_data                                   |
+---+--------------------------------------------+
|1  |{"user_id": 100, "features": {"score":0.8}} |
+---+--------------------------------------------+
```
<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Define a schema matching the JSON structure</span>
<span style="color: #CCC">schema</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">StructType([</span>
    <span style="color: #CCC">StructField(</span><span style="color: #7EC699">"user_id"</span><span style="color: #CCC">,</span> <span style="color: #CCC">LongType()),</span>
    <span style="color: #CCC">StructField(</span><span style="color: #7EC699">"features"</span><span style="color: #CCC">,</span> <span style="color: #CCC">MapType(StringType(),</span> <span style="color: #CCC">DoubleType()))</span>
<span style="color: #CCC">])</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #999"># Convert JSON strings to a struct type</span>
<span style="color: #CCC">df2</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">withColumn(</span><span style="color: #7EC699">"parsed"</span><span style="color: #CCC">,</span> <span style="color: #CCC">from_json(col(</span><span style="color: #7EC699">"json_data"</span><span style="color: #CCC">),</span> <span style="color: #CCC">schema))</span></code></pre>

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128161;</span>
<div>
<strong style="color: #00695c; font-size: 1.1em">Don't Know the Schema?</strong>
<p style="margin: 8px 0 0 0; color: #333">Use <code>schema_of_json()</code> on a representative sample value to infer a schema, or use the Spark 4 <code>VARIANT</code> data type (<code>parse_json()</code>) to store semi-structured data without declaring a schema at all - <code>VARIANT</code> keeps the full structure queryable with dot notation while tolerating schema drift.</p>
</div>
</div>
</div>



## C. Working with Struct Types

<p style="font-size: 1em; line-height: 1.6; color: #333">Struct types maintain a logical grouping of related fields - like a table within a column:</p>

- Struct fields can be accessed and operated on using dot notation or the `getField()` column function

Given input data with struct column 'user':

```
user: struct&lt;name:string, age:int, scores:array&lt;double&gt;&gt;;
```

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Accessing struct fields</span>
<span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span>
    <span style="color: #CCC">col(</span><span style="color: #7EC699">"user.name"</span><span style="color: #CCC">),</span> <span style="color: #999"># Direct field access</span>
    <span style="color: #CCC">col(</span><span style="color: #7EC699">"user"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">getField(</span><span style="color: #7EC699">"age"</span><span style="color: #CCC">),</span> <span style="color: #999"># Alternative access</span>
    <span style="color: #CCC">col(</span><span style="color: #7EC699">"user.scores"</span><span style="color: #CCC">)[</span><span style="color: #F08D49">0</span><span style="color: #CCC">]</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"first_score"</span><span style="color: #CCC">)</span> <span style="color: #999"># Nested access</span>
<span style="color: #CCC">)</span></code></pre>



## D. The explode() Function

<p style="font-size: 1em; line-height: 1.6; color: #333">Unnesting arrays while preserving other columns - one row per array element:</p>

Input: DataFrame with array column:

```
+---+----------------+
|id |items           |
+---+----------------+
|1  |["a", "b", "c"] |
|2  |["x", "y"]      |
+---+----------------+
```

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span><span style="color: #7EC699">"id"</span><span style="color: #CCC">,</span> <span style="color: #CCC">explode(</span><span style="color: #7EC699">"items"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"item"</span><span style="color: #CCC">))</span></code></pre>

Results in...

```
+---+----+
|id |item|
+---+----+
|1  |a   |
|1  |b   |
|1  |c   |
|2  |x   |
|2  |y   |
+---+----+
```



## E. Common Array Column Operations

Common functions for array column manipulation:

| Function | Purpose | Example |
|----------|---------|---------|
| __`array_contains(col, val)`__ | Tests if the array contains a specific value | `array_contains(items, "a")` |
| __`size(col)`__ / __`array_size(col)`__ | Returns the number of elements | `array_size(items)` |
| __`element_at(col, n)`__ | Returns the nth element (1-based indexing) | `element_at(items, 2)` |
| __`array_distinct(col)`__ | Removes duplicates from an array column | `array_distinct(items)` |

## F. Aggregating to Collections

<p style="font-size: 1em; line-height: 1.6; color: #333">Collection aggregates are the inverse of <code>explode()</code> - they gather rows into arrays:</p>

- `collect_list()` is an aggregate function that gathers all values from a column into an array - commonly used with `groupBy()` to build arrays of related values within each group
- `collect_set()` works similarly but removes duplicates, producing arrays of unique values

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Get a list (array) of products for each region</span>
<span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">groupBy(</span><span style="color: #7EC699">"region"</span><span style="color: #CCC">)</span> \
  <span style="color: #67CDCC">.</span><span style="color: #CCC">agg(</span>
    <span style="color: #CCC">collect_list(</span><span style="color: #7EC699">"product"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"all_products"</span><span style="color: #CCC">),</span>    <span style="color: #999"># keeps duplicates</span>
    <span style="color: #CCC">collect_set(</span><span style="color: #7EC699">"product"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"unique_products"</span><span style="color: #CCC">)</span>   <span style="color: #999"># unique only</span>
  <span style="color: #CCC">)</span></code></pre>



## G. Best Practices for Complex Data Types

<p style="font-size: 1em; line-height: 1.6; color: #333">Considerations when working with nested structures:</p>

- `explode()` on large arrays can cause data explosion - 1 row becomes N rows, which can overwhelm memory and processing
- `collect_list()` / `collect_set()` can be memory-intensive when groups are large, as they hold all values per group in memory
- Consider `collect_set()` when duplicates aren't needed and order doesn't matter:
  - Uses less memory because it removes duplicates
  - Less shuffle overhead as duplicate values don't need to be collected

## Summary

- Arrays, structs, and maps let one column carry structure; they are the natural shape of JSON and event data
- Convert JSON strings to structs early with `from_json()` - structs are typed, compact, and optimizer-friendly; Spark 4 `VARIANT` covers the schema-unknown case
- `explode()` turns array elements into rows; `collect_list()` and `collect_set()` do the reverse
- Watch the memory cost: exploding large arrays multiplies rows, and collection aggregates hold whole groups in memory

## Knowledge Check

Answer the questions below to check your understanding of this lecture.

<div id="mcqRoot"></div>

<script>
(function() {
    var questions = [
        {
                "question": "When working with complex data types in Spark, which statement is correct about struct types?",
                "options": [
                        {
                                "id": "a",
                                "text": "Struct fields can only be accessed using getField()"
                        },
                        {
                                "id": "b",
                                "text": "Struct types require JSON parsing at runtime"
                        },
                        {
                                "id": "c",
                                "text": "Struct types enforce schema and type safety"
                        },
                        {
                                "id": "d",
                                "text": "Struct fields must not be predefined, like Maps"
                        }
                ],
                "correct": "c",
                "explanation": "A struct has predefined, named, typed fields - that is exactly what distinguishes it from a JSON string (parsed at runtime) and from a map (whose keys are not predefined). Fields are accessible with dot notation or getField()."
        },
        {
                "question": "Which operation can cause significant performance issues when working with large datasets?",
                "options": [
                        {
                                "id": "a",
                                "text": "Using array_contains() on array columns"
                        },
                        {
                                "id": "b",
                                "text": "Converting JSON to struct types at ingestion"
                        },
                        {
                                "id": "c",
                                "text": "Using explode() on large array columns"
                        },
                        {
                                "id": "d",
                                "text": "Accessing struct fields with dot notation"
                        }
                ],
                "correct": "c",
                "explanation": "explode() multiplies rows - one row per array element. On large arrays this data explosion can overwhelm memory and processing. The other operations are cheap or actively beneficial."
        },
        {
                "question": "When aggregating data into arrays in Spark, which statement is true?",
                "options": [
                        {
                                "id": "a",
                                "text": "collect_set() is always more efficient than collect_list() regardless of data"
                        },
                        {
                                "id": "b",
                                "text": "array_agg() must be used instead of collect_list()"
                        },
                        {
                                "id": "c",
                                "text": "collect_list() keeps duplicates while collect_set() removes them"
                        },
                        {
                                "id": "d",
                                "text": "collect_set() requires manual duplicate removal"
                        }
                ],
                "correct": "c",
                "explanation": "collect_list() gathers every value including duplicates; collect_set() produces unique values only, which also tends to use less memory and shuffle less data."
        },
        {
                "question": "Your pipeline receives JSON payloads whose structure changes frequently. Which Spark 4 feature stores them without declaring a schema up front?",
                "options": [
                        {
                                "id": "a",
                                "text": "The VARIANT data type with parse_json()"
                        },
                        {
                                "id": "b",
                                "text": "MapType with string keys"
                        },
                        {
                                "id": "c",
                                "text": "schema_of_json() on every row"
                        },
                        {
                                "id": "d",
                                "text": "Storing the payload as a plain string"
                        }
                ],
                "correct": "a",
                "explanation": "VARIANT (new in Spark 4) stores semi-structured data in an efficient binary form that remains queryable with dot notation and tolerates schema drift. A plain string would require parsing on every read, and maps lose the nested structure's types."
        }
];
    
    var currentIndex = 0;
    var score = 0;
    var selectedAnswer = null;
    var showingFeedback = false;
    
    function init() {
        currentIndex = 0;
        score = 0;
        selectedAnswer = null;
        showingFeedback = false;
        render();
    }
    
    function render() {
        var root = document.getElementById('mcqRoot');
        var html = '';
        var isComplete = currentIndex >= questions.length;
        
        html += '<div style="font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; width: 100%; margin: 10px 0; padding: 24px; background: #f5f7fa; border-radius: 12px; border: 1px solid #e0e0e0; box-sizing: border-box;">';
        
        // Header
        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">';
        html += '<div>';
        html += '<div style="font-size: 1.4em; font-weight: 600; color: #333;">📝 Multiple Choice Quiz</div>';
        html += '<div style="font-size: 0.95em; color: #666; margin-top: 4px;">Knowledge Check</div>';
        html += '</div>';
        html += '<div style="display: flex; gap: 12px; align-items: center;">';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e8f5e9; color: #2e7d32;">Score: ' + score + '/' + questions.length + '</span>';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e3f2fd; color: #1976d2;">Q' + (currentIndex + 1) + '/' + questions.length + '</span>';
        html += '</div></div>';
        
        if (isComplete) {
            // Results
            var percentage = Math.round(score / questions.length * 100);
            html += '<div style="text-align: center; padding: 40px 20px; background: #fff; border-radius: 12px; border: 2px solid #e0e0e0;">';
            html += '<div style="font-size: 3em; margin-bottom: 16px;">' + (percentage >= 75 ? '🏆' : percentage >= 50 ? '👍' : '📚') + '</div>';
            html += '<div style="font-size: 1.8em; font-weight: 700; color: #333; margin-bottom: 8px;">Quiz Complete!</div>';
            html += '<div style="font-size: 1.2em; color: #666; margin-bottom: 24px;">You scored ' + score + ' out of ' + questions.length + ' (' + percentage + '%)</div>';
            html += '<button id="retakeBtn" style="padding: 14px 32px; background: #1976d2; color: white; border: none; border-radius: 8px; cursor: pointer; font-size: 1.1em; font-weight: 600;">Retake Quiz</button>';
            html += '</div>';
        } else {
            var q = questions[currentIndex];
            
            // Question
            html += '<div style="background: #fff; border-radius: 12px; border: 2px solid #e0e0e0; padding: 24px; margin-bottom: 20px;">';
            html += '<div style="font-size: 1.15em; color: #333; line-height: 1.6;">' + q.question + '</div>';
            html += '</div>';
            
            // Options
            html += '<div style="display: flex; flex-direction: column; gap: 12px; margin-bottom: 20px;">';
            
            q.options.forEach(function(opt) {
                var isSelected = selectedAnswer === opt.id;
                var isCorrect = opt.id === q.correct;
                var bgColor = '#fff';
                var borderColor = '#e0e0e0';
                var textColor = '#333';
                
                if (showingFeedback) {
                    if (isCorrect) {
                        bgColor = '#e8f5e9';
                        borderColor = '#4caf50';
                        textColor = '#2e7d32';
                    } else if (isSelected && !isCorrect) {
                        bgColor = '#ffebee';
                        borderColor = '#f44336';
                        textColor = '#c62828';
                    }
                } else if (isSelected) {
                    bgColor = '#e3f2fd';
                    borderColor = '#1976d2';
                    textColor = '#1565c0';
                }
                
                var cursor = showingFeedback ? 'default' : 'pointer';
                
                html += '<div class="mcq-option" data-id="' + opt.id + '" style="display: flex; align-items: center; gap: 16px; padding: 16px 20px; background: ' + bgColor + '; border: 2px solid ' + borderColor + '; border-radius: 8px; cursor: ' + cursor + '; transition: all 0.15s ease;">';
                html += '<div style="width: 32px; height: 32px; border: 2px solid ' + borderColor + '; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: 700; color: ' + textColor + '; background: ' + (isSelected || (showingFeedback && isCorrect) ? borderColor : 'transparent') + '; color: ' + (isSelected || (showingFeedback && isCorrect) ? '#fff' : textColor) + ';">' + opt.id.toUpperCase() + '</div>';
                html += '<div style="flex: 1; font-size: 1em; color: ' + textColor + ';">' + opt.text + '</div>';
                if (showingFeedback && isCorrect) {
                    html += '<span style="color: #4caf50; font-size: 1.2em;">✓</span>';
                } else if (showingFeedback && isSelected && !isCorrect) {
                    html += '<span style="color: #f44336; font-size: 1.2em;">✗</span>';
                }
                html += '</div>';
            });
            
            html += '</div>';
            
            // Feedback
            if (showingFeedback) {
                var isCorrect = selectedAnswer === q.correct;
                html += '<div style="background: ' + (isCorrect ? '#e8f5e9' : '#fff3e0') + '; border: 2px solid ' + (isCorrect ? '#4caf50' : '#ff9800') + '; border-radius: 8px; padding: 16px 20px; margin-bottom: 20px;">';
                html += '<div style="font-weight: 600; color: ' + (isCorrect ? '#2e7d32' : '#e65100') + '; margin-bottom: 8px;">' + (isCorrect ? '✓ Correct!' : '✗ Not quite') + '</div>';
                html += '<div style="color: #333; font-size: 0.95em; line-height: 1.5;">' + q.explanation + '</div>';
                html += '</div>';
            }
            
            // Buttons
            html += '<div style="display: flex; gap: 12px; justify-content: flex-end;">';
            if (!showingFeedback) {
                html += '<button id="submitBtn" style="padding: 12px 28px; background: ' + (selectedAnswer ? '#4caf50' : '#ccc') + '; color: white; border: none; border-radius: 6px; cursor: ' + (selectedAnswer ? 'pointer' : 'not-allowed') + '; font-size: 1em; font-weight: 600;"' + (selectedAnswer ? '' : ' disabled') + '>Submit Answer</button>';
            } else {
                html += '<button id="nextQBtn" style="padding: 12px 28px; background: #1976d2; color: white; border: none; border-radius: 6px; cursor: pointer; font-size: 1em; font-weight: 600;">' + (currentIndex < questions.length - 1 ? 'Next Question' : 'See Results') + '</button>';
            }
            html += '</div>';
        }
        
        html += '</div>';
        
        root.innerHTML = html;
        attachEvents();
    }
    
    function attachEvents() {
        var options = document.querySelectorAll('.mcq-option');
        var submitBtn = document.getElementById('submitBtn');
        var nextQBtn = document.getElementById('nextQBtn');
        var retakeBtn = document.getElementById('retakeBtn');
        
        options.forEach(function(opt) {
            opt.addEventListener('click', function() {
                if (!showingFeedback) {
                    selectedAnswer = this.dataset.id;
                    render();
                }
            });
            
            if (!showingFeedback) {
                opt.addEventListener('mouseenter', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#1976d2';
                        this.style.background = '#f5f9ff';
                    }
                });
                opt.addEventListener('mouseleave', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#e0e0e0';
                        this.style.background = '#fff';
                    }
                });
            }
        });
        
        if (submitBtn) {
            submitBtn.addEventListener('click', function() {
                if (selectedAnswer) {
                    var q = questions[currentIndex];
                    if (selectedAnswer === q.correct) {
                        score++;
                    }
                    showingFeedback = true;
                    render();
                }
            });
        }
        
        if (nextQBtn) {
            nextQBtn.addEventListener('click', function() {
                currentIndex++;
                selectedAnswer = null;
                showingFeedback = false;
                render();
            });
        }
        
        if (retakeBtn) {
            retakeBtn.addEventListener('click', function() {
                init();
            });
        }
    }
    
    if (typeof NodeList.prototype.forEach !== 'function') {
        NodeList.prototype.forEach = Array.prototype.forEach;
    }
    
    init();
})();
</script>

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>